# Day 2 — Advanced RAG 평가 실습

목표: KorQuAD(위키)와 KLUE-MRC(뉴스)에서 Naive RAG와 Advanced RAG를 비교하고, Langfuse로 실행 흐름을 관측한다.

**완료 조건:** 두 도메인의 RAGAS 비교표가 출력되고, Langfuse Traces에서 `rag-question`과 `ragas-evaluation`을 확인한다.


## 0. 로컬 환경과 `.env`

이 폴더의 `pyproject.toml`에 아래 의존성을 추가한 뒤 `uv sync`를 실행한다. Python은 **64비트 3.11**을 사용한다.

```powershell
uv add jupyter ipykernel python-dotenv nest-asyncio datasets pandas tiktoken chromadb sentence-transformers langfuse "ragas==0.2.10" "langchain==0.2.17" "langchain-core==0.2.43" "langchain-community==0.2.19" "langchain-openai==0.1.25" "langchain-text-splitters==0.2.4" "langchain-chroma==0.1.4"
uv add --upgrade torch --torch-backend=auto
uv sync --torch-backend=auto
uv run python -m ipykernel install --user --name advanced-rag-eval --display-name "Advanced RAG Eval"
```

노트북과 같은 폴더의 `.env`에는 아래 값을 둔다. `LANGFUSE_HOST`는 로컬 Docker Langfuse용이고, Cloud를 쓴다면 Cloud URL을 넣는다.

```env
OPENAI_API_KEY=...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=http://localhost:3000
```


## 1. 설정과 Langfuse 초기화

Langfuse는 여기서 먼저 초기화한다. 이후 모든 LangChain LLM 호출은 callback으로, 각 RAG 문항과 RAGAS 평가는 상위 trace로 자동 기록된다.


In [1]:
# Load environment variables and define the experiment-wide configuration.
# 환경 변수를 읽고 실험 전체에서 사용할 설정을 정의합니다.
import os
from contextlib import nullcontext
from pathlib import Path

from dotenv import load_dotenv

# Read API keys and local endpoints from the project .env file.
# 프로젝트 .env 파일에서 API 키와 로컬 엔드포인트를 읽습니다.
load_dotenv()
# Disable Chroma anonymous telemetry before Chroma is imported later.
# 이후 Chroma를 불러오기 전에 익명 텔레메트리를 비활성화합니다.
os.environ["ANONYMIZED_TELEMETRY"] = "FALSE"
# Support either LANGFUSE_HOST (local Docker) or LANGFUSE_BASE_URL.
# LANGFUSE_HOST(로컬 Docker)와 LANGFUSE_BASE_URL 모두를 지원합니다.
if os.getenv("LANGFUSE_HOST") and not os.getenv("LANGFUSE_BASE_URL"):
    os.environ["LANGFUSE_BASE_URL"] = os.environ["LANGFUSE_HOST"]
os.environ.setdefault("LANGFUSE_TRACING_ENVIRONMENT", "development")

# Stop early with a clear message when a required credential is missing.
# 필수 인증 정보가 없으면 명확한 메시지와 함께 초기에 중단합니다.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env."
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "Set LANGFUSE_PUBLIC_KEY in .env."
assert os.getenv("LANGFUSE_SECRET_KEY"), "Set LANGFUSE_SECRET_KEY in .env."

# Set artifact locations, sample sizes, random seed, and model identifiers.
# 결과 저장 경로, 표본 수, 난수 시드, 모델 식별자를 설정합니다.
WORK_DIR = Path("./advanced-rag-artifacts")
WORK_DIR.mkdir(exist_ok=True)
CORPUS_N = 500
EVAL_N = 20
SEED = 42
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

# Create the Langfuse client and LangChain callback used by later LLM calls.
# 이후 LLM 호출을 기록할 Langfuse 클라이언트와 LangChain 콜백을 생성합니다.
from langfuse import get_client, propagate_attributes
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()
print("Langfuse:", os.environ["LANGFUSE_BASE_URL"])
print(f"Corpus={CORPUS_N}, evaluation questions={EVAL_N}")


Langfuse: http://localhost:3000
Corpus=500, evaluation questions=20


## 2. 공통 도구와 프롬프트

Chroma telemetry는 명시적으로 끈다. 이것은 Langfuse trace와 무관한 Chroma 익명 통계 오류를 막기 위한 설정이다.


In [2]:
# Import RAG, dataset, vector-store, evaluation-support, and reranking components.
# RAG, 데이터셋, 벡터 저장소, 평가 지원, 리랭킹 구성 요소를 불러옵니다.
from collections import defaultdict
from uuid import uuid4

import nest_asyncio
import pandas as pd
import tiktoken
from chromadb.config import Settings
from datasets import Dataset, load_dataset
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

# Allow notebook cells to run async-compatible libraries without an event-loop conflict.
# 노트북의 이벤트 루프 충돌 없이 비동기 호환 라이브러리를 실행할 수 있게 합니다.
nest_asyncio.apply()
# Create shared LLM, embedding, Chroma, and token-counting utilities.
# 공통으로 사용할 LLM, 임베딩, Chroma, 토큰 계산 도구를 생성합니다.
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
chroma_settings = Settings(anonymized_telemetry=False)
tokenizer = tiktoken.get_encoding("cl100k_base")

# Define prompts for answer generation, query expansion, HyDE, and Self-RAG decisions.
# 답변 생성, 질의 확장, HyDE, Self-RAG 판단용 프롬프트를 정의합니다.
RAG_PROMPT = ChatPromptTemplate.from_template(
    "Use only the documents below. Answer the question concisely in Korean. "
    "If the evidence is insufficient, say that you do not know.\n\n"
    "[Documents]\n{context}\n\n[Question]\n{question}\n\n[Answer]"
)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "Create four Korean search queries with the same intent as the question. "
    "Return only one query per line, with no numbering or explanation.\n\nQuestion: {question}"
)
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "Write one plausible Korean reference paragraph that could answer the question. "
    "Do not claim uncertain facts as certain. Return only the paragraph.\n\nQuestion: {question}"
)
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "Return YES if answering the question requires a specific document, current fact, proper noun, number, quote, or domain knowledge. "
    "Return NO only when general knowledge, a simple calculation, or a definition is enough. Return exactly YES or NO.\n\nQuestion: {question}"
)
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "Return SUPPORTED only when every factual claim in the answer is supported by the documents. "
    "Return NOT_SUPPORTED if any claim is missing, exaggerated, or contradictory. Return exactly one label.\n\n"
    "[Documents]\n{context}\n\n[Answer]\n{answer}"
)

# Build a common Langfuse callback configuration for each LLM stage.
# 각 LLM 단계에서 공통으로 사용할 Langfuse 콜백 설정을 만듭니다.
def trace_config(stage, domain):
    return {
        "callbacks": [langfuse_handler],
        "metadata": {"langfuse_tags": ["advanced-rag-evaluation", stage], "domain": domain},
    }

# Invoke a LangChain runnable while recording the stage in Langfuse.
# Langfuse에 단계를 기록하면서 LangChain 실행 체인을 호출합니다.
def invoke_chain(chain, payload, stage, domain):
    return chain.invoke(payload, config=trace_config(stage, domain))

# Create a manual Langfuse observation for a full question or evaluation run.
# 질문 하나 또는 평가 실행 전체를 위한 Langfuse 관측치를 만듭니다.
def trace_observation(name, input_data, metadata):
    return langfuse.start_as_current_observation(
        as_type="chain", name=name, input=input_data, metadata=metadata
    )

# Count tokens so text splitting uses the same token unit as the LLM ecosystem.
# 텍스트 분할이 LLM 생태계와 같은 토큰 단위를 사용하도록 토큰 수를 계산합니다.
def token_len(text):
    return len(tokenizer.encode(text))

# Concatenate retrieved documents into the context supplied to the answer prompt.
# 검색된 문서를 답변 프롬프트에 넣을 컨텍스트 문자열로 합칩니다.
def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

# Generate a grounded Korean answer using only the retrieved documents.
# 검색된 문서만 근거로 사용해 한국어 답변을 생성합니다.
def answer_from_docs(question, documents, domain):
    return invoke_chain(
        RAG_PROMPT | llm | StrOutputParser(),
        {"question": question, "context": format_docs(documents)},
        stage="answer-generation",
        domain=domain,
    )


## 3. KorQuAD와 KLUE-MRC 로드 및 인덱스 생성

답이 있는 validation 샘플만 사용한다. 각 도메인은 별도의 Chroma collection으로 저장돼 서로 섞이지 않는다.


In [3]:
# Load a reproducible KorQuAD subset and normalize it to the shared record schema.
# 재현 가능한 KorQuAD 부분집합을 불러와 공통 레코드 형식으로 정규화합니다.
def load_korquad_records(limit=CORPUS_N, seed=SEED):
    ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=seed).select(range(limit))
    return [
        {"question": ex["question"], "context": ex["context"], "answer": ex["answers"]["text"][0], "title": ex["title"]}
        for ex in ds if ex["answers"]["text"]
    ]

# Load answerable KLUE-MRC news examples and normalize them to the same schema.
# 답변 가능한 KLUE-MRC 뉴스 예제를 불러와 같은 형식으로 정규화합니다.
def load_klue_records(limit=CORPUS_N, seed=SEED):
    ds = load_dataset("klue", "mrc", split="validation")
    ds = ds.filter(lambda ex: not ex["is_impossible"] and len(ex["answers"]["text"]) > 0)
    ds = ds.shuffle(seed=seed).select(range(limit))
    return [
        {"question": ex["question"], "context": ex["context"], "answer": ex["answers"]["text"][0], "title": ex["title"]}
        for ex in ds
    ]

# Deduplicate contexts, split them into token-aware chunks, and index them in Chroma.
# 중복 문맥을 제거하고 토큰 기준 청크로 나눈 뒤 Chroma에 색인합니다.
def build_vectorstore(records, domain):
    unique_contexts = {}
    for record in records:
        unique_contexts.setdefault(record["context"], record["title"])
    source_docs = [Document(page_content=context, metadata={"title": title, "domain": domain})
                   for context, title in unique_contexts.items()]
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=80, length_function=token_len
    ).split_documents(source_docs)
    db = Chroma(
        collection_name=f"{domain}_{uuid4().hex[:8]}",
        embedding_function=embedding,
        client_settings=chroma_settings,
        persist_directory=str(WORK_DIR / "chroma"),
    )
    for start in range(0, len(chunks), 100):
        db.add_documents(chunks[start:start + 100])
    print(f"{domain}: source documents={len(source_docs)}, chunks={len(chunks)}")
    return db

# Build independent corpus indexes so each domain is evaluated only against its own documents.
# 각 도메인이 자신의 문서만 대상으로 평가되도록 독립적인 코퍼스 인덱스를 만듭니다.
records_korquad = load_korquad_records()
records_klue = load_klue_records()
db_korquad = build_vectorstore(records_korquad, "korquad")
db_klue = build_vectorstore(records_klue, "klue")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


korquad: source documents=390, chunks=586


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


klue: source documents=499, chunks=1468


## 4. Reranker 로드

처음 실행에서 `BAAI/bge-reranker-v2-m3`를 다운로드한다. 다운로드 시간은 CUDA 여부와 별개이며, CUDA는 다운로드 후 reranking 추론 속도에 영향을 준다.


In [4]:
# Load the cross-encoder reranker on GPU when PyTorch CUDA is available.
# PyTorch CUDA가 가능하면 GPU에 Cross-Encoder 리랭커를 불러옵니다.
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder(RERANKER_MODEL, device=device)
print("Reranker device:", reranker.device)


Reranker device: cuda:0


## 5. Naive RAG와 Advanced RAG 정의

Advanced 검색 흐름은 `Multi-Query → RRF → HyDE 추가 검색 → cross-encoder rerank`다. Self-RAG는 검색 필요성 판단과 답변 근거성 비평을 더한다. 평가에서는 공정한 문서 기반 비교를 위해 검색을 강제한다.


In [5]:
# Fuse ranked lists with Reciprocal Rank Fusion (RRF); higher repeated ranks receive more score.
# RRF로 여러 순위 목록을 합치며, 여러 검색에서 상위에 반복된 문서에 더 높은 점수를 줍니다.
def reciprocal_rank_fusion(results_per_query, k=60, top_k=10):
    scores = defaultdict(float)
    docs_by_key = {}
    for documents in results_per_query:
        for rank, document in enumerate(documents, start=1):
            key = document.page_content
            scores[key] += 1.0 / (k + rank)
            docs_by_key[key] = document
    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [docs_by_key[key] for key in ranked_keys[:top_k]]

# Generate query paraphrases to retrieve documents expressed with different wording.
# 다른 표현으로 작성된 문서를 찾기 위해 질의 패러프레이즈를 생성합니다.
def fan_out_queries(question, domain):
    raw = invoke_chain(SUBQUERY_PROMPT | llm | StrOutputParser(), {"question": question}, "multi-query", domain)
    variants = [line.strip(" -0123456789. ") for line in raw.splitlines() if line.strip()]
    return [question] + variants[:4]

# Generate a hypothetical reference passage and use it as an additional retrieval query.
# 가상의 참고 문단을 생성하고 추가 검색 질의로 사용합니다.
def hyde_retrieve(question, db, domain, top_k=8):
    hypothetical = invoke_chain(HYDE_PROMPT | llm | StrOutputParser(), {"question": question}, "hyde", domain)
    return db.similarity_search(hypothetical, k=top_k), hypothetical

# Score query-document pairs with a cross-encoder and retain the best final context.
# Cross-Encoder로 질문-문서 쌍을 점수화해 최종 컨텍스트를 선별합니다.
def rerank(question, documents, top_k=3):
    scores = reranker.predict([(question, doc.page_content) for doc in documents])
    ranked = sorted(zip(documents, scores), key=lambda pair: pair[1], reverse=True)
    top_ranked = ranked[:top_k]
    # Return both documents and scores so Langfuse can explain the reranking decision.
    # Langfuse에서 리랭킹 판단 근거를 확인할 수 있도록 문서와 점수를 함께 반환합니다.
    return [doc for doc, _ in top_ranked], [float(score) for _, score in top_ranked]

# Baseline: retrieve the three nearest chunks with the original question only.
# 베이스라인: 원본 질문만으로 가장 가까운 청크 3개를 검색합니다.
def naive_rag(question, db, domain):
    docs = db.similarity_search(question, k=3)
    return answer_from_docs(question, docs, domain), docs

# Advanced retrieval: Multi-Query + HyDE -> RRF -> cross-encoder reranking.
# 고급 검색: Multi-Query와 HyDE 결과를 RRF로 합친 뒤 Cross-Encoder로 재정렬합니다.
def advanced_retrieve(question, db, domain, candidate_k=8, final_k=3):
    queries = fan_out_queries(question, domain)
    result_lists = [db.similarity_search(query, k=candidate_k) for query in queries]
    hyde_docs, hypothetical = hyde_retrieve(question, db, domain, top_k=candidate_k)
    fused_candidates = reciprocal_rank_fusion(result_lists + [hyde_docs], top_k=10)
    top_docs, reranker_scores = rerank(question, fused_candidates, top_k=final_k)
    # Keep retrieval diagnostics for question-level inspection in Langfuse.
    # Langfuse에서 질문별 검색 과정을 점검할 수 있도록 진단 정보를 보관합니다.
    return top_docs, {"queries": queries, "hypothetical": hypothetical, "candidate_count": len(fused_candidates), "reranker_top_scores": reranker_scores}

# Self-RAG: decide whether retrieval is needed, critique grounding, then retry once if needed.
# Self-RAG: 검색 필요성을 판단하고 근거성을 비평한 뒤 필요하면 한 번 재시도합니다.
def self_rag(question, db, domain, max_retries=1, force_retrieval=True):
    decision = "YES" if force_retrieval else invoke_chain(
        RETRIEVE_DECISION_PROMPT | llm | StrOutputParser(), {"question": question}, "retrieve-decision", domain
    ).strip().upper()
    if decision.startswith("NO"):
        answer = llm.invoke(question, config=trace_config("closed-book-answer", domain)).content
        return answer, [], {"decision": decision, "critique": "SKIPPED", "retries": 0}
    docs, trace = advanced_retrieve(question, db, domain)
    for attempt in range(max_retries + 1):
        answer = answer_from_docs(question, docs, domain)
        critique = invoke_chain(
            CRITIQUE_PROMPT | llm | StrOutputParser(),
            {"context": format_docs(docs), "answer": answer},
            "self-rag-critique", domain,
        ).strip().upper()
        if critique.startswith("SUPPORTED") or attempt == max_retries:
            return answer, docs, {"decision": decision, "critique": critique, "retries": attempt, **trace}
        docs, trace = advanced_retrieve(question, db, domain, candidate_k=12, final_k=3)


## 6. 두 파이프라인 실행

각 질문은 Langfuse의 `rag-question` trace 하나로 기록된다. 내부에 Multi-Query, HyDE, 답변 생성, 비평 LLM 호출이 하위 generation으로 연결된다.


In [6]:
# Run one pipeline over the evaluation questions and record each question as a Langfuse trace.
# 평가 질문 전체에 파이프라인을 실행하고 각 질문을 Langfuse trace로 기록합니다.
def run_pipeline(records, db, domain, pipeline):
    outputs = []
    for record in records[:EVAL_N]:
        # Add filterable pipeline/domain tags to observations created for this question.
        # 이 질문에서 생성되는 관측치에 파이프라인/도메인 필터 태그를 붙입니다.
        with propagate_attributes(tags=["advanced-rag-evaluation", pipeline, domain], session_id=f"rag-eval-{domain}"):
            with trace_observation(
                "rag-question",
                {"question": record["question"]},
                {"pipeline": pipeline, "domain": domain},
            ) as span:
                if pipeline == "naive":
                    answer, docs = naive_rag(record["question"], db, domain)
                    trace = {}
                elif pipeline == "advanced":
                    answer, docs, trace = self_rag(record["question"], db, domain, force_retrieval=True)
                else:
                    raise ValueError("pipeline must be naive or advanced")
                # Capture the active trace ID for per-question RAGAS score attachment.
                # 질문별 RAGAS 점수를 붙일 수 있도록 활성 trace ID를 저장합니다.
                trace_id = langfuse.get_current_trace_id()
                retrieved_contexts = [doc.page_content for doc in docs]
                # Store evidence and reranker diagnostics, not only a document count.
                # 문서 개수뿐 아니라 근거 문서 원문과 리랭커 진단 정보를 저장합니다.
                span.update(output={
                    "answer": answer,
                    "reference": record["answer"],
                    "retrieved_contexts": retrieved_contexts,
                    "retrieved_document_count": len(docs),
                    "self_rag": trace,
                })
        outputs.append({
            "user_input": record["question"],
            "response": answer,
            "retrieved_contexts": retrieved_contexts,
            "reference": record["answer"],
            "trace_id": trace_id,
            "pipeline": pipeline,
            "domain": domain,
        })
    return outputs

# Convert pipeline outputs to the schema expected by RAGAS.
# 파이프라인 출력을 RAGAS가 요구하는 데이터 형식으로 변환합니다.
def outputs_to_dataset(outputs):
    # Keep only the columns accepted by RAGAS; trace metadata stays in the output list.
    # RAGAS가 받는 열만 전달하고 trace 메타데이터는 출력 목록에 유지합니다.
    ragas_keys = ["user_input", "response", "retrieved_contexts", "reference"]
    return Dataset.from_dict({key: [row[key] for row in outputs] for key in ragas_keys})

# Execute Naive and Advanced RAG on the same questions in both domains.
# 두 도메인의 동일한 질문에 Naive와 Advanced RAG를 모두 실행합니다.
naive_korquad = run_pipeline(records_korquad, db_korquad, "korquad", "naive")
advanced_korquad = run_pipeline(records_korquad, db_korquad, "korquad", "advanced")
naive_klue = run_pipeline(records_klue, db_klue, "klue", "naive")
advanced_klue = run_pipeline(records_klue, db_klue, "klue", "advanced")

# Prepare the four result sets for paired RAGAS evaluation.
# 쌍을 이룬 RAGAS 평가를 위해 네 개의 결과 집합을 준비합니다.
naive_ds_korquad, advanced_ds_korquad = outputs_to_dataset(naive_korquad), outputs_to_dataset(advanced_korquad)
naive_ds_klue, advanced_ds_klue = outputs_to_dataset(naive_klue), outputs_to_dataset(advanced_klue)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


## 7. RAGAS 평가와 비교표

지표는 검색 품질(Context Precision/Recall), 문서 근거성(Faithfulness), 답변 적합성(Answer Relevancy)이다. 각 도메인의 평가 전체는 Langfuse에서 `ragas-evaluation` trace로 남는다. 또한 평가가 끝나면 질문별 RAGAS 점수를 원래의 `rag-question` trace에 붙여, Langfuse에서 낮은 점수 사례부터 정렬해 확인할 수 있다.


In [7]:
# Import RAGAS metrics covering grounding, answer fit, and retrieval quality.
# 근거성, 답변 적합성, 검색 품질을 측정하는 RAGAS 지표를 불러옵니다.
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# Configure the judge LLM and embeddings used internally by RAGAS.
# RAGAS 내부 평가에 사용할 심사 LLM과 임베딩 모델을 설정합니다.
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
judge_llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
judge_embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Evaluate Naive and Advanced outputs for one domain and log their means to Langfuse.
# 한 도메인의 Naive와 Advanced 출력을 평가하고 평균 점수를 Langfuse에 기록합니다.
def evaluate_pair(naive_ds, advanced_ds, domain):
    with trace_observation(
        "ragas-evaluation",
        {"rows": len(naive_ds), "metrics": metric_names},
        {"domain": domain, "stage": "evaluation"},
    ) as span:
        naive_df = evaluate(naive_ds, metrics=metrics, llm=judge_llm, embeddings=judge_embeddings, raise_exceptions=False).to_pandas()
        advanced_df = evaluate(advanced_ds, metrics=metrics, llm=judge_llm, embeddings=judge_embeddings, raise_exceptions=False).to_pandas()
        span.update(output={
            "naive_means": naive_df[metric_names].mean().to_dict(),
            "advanced_means": advanced_df[metric_names].mean().to_dict(),
        })
    return naive_df, advanced_df

# Convert per-question scores into one row per metric for a readable comparison table.
# 질문별 점수를 지표당 한 행으로 바꿔 읽기 쉬운 비교표를 만듭니다.
def summarize_pair(naive_df, advanced_df, domain):
    return pd.DataFrame([
        {"domain": domain, "metric": metric, "naive": naive_df[metric].mean(),
         "advanced": advanced_df[metric].mean(), "delta": advanced_df[metric].mean() - naive_df[metric].mean()}
        for metric in metric_names
    ])

# Attach each RAGAS metric to its original question trace for score-based filtering in Langfuse.
# Langfuse에서 점수 기반 필터링을 할 수 있도록 각 RAGAS 지표를 원래 질문 trace에 붙입니다.
def attach_ragas_scores(outputs, score_df):
    score_count = 0
    for output, (_, score_row) in zip(outputs, score_df.iterrows()):
        for metric in metric_names:
            score = score_row[metric]
            if pd.notna(score):
                langfuse.create_score(
                    name=metric,
                    value=float(score),
                    trace_id=output["trace_id"],
                    data_type="NUMERIC",
                    metadata={"domain": output["domain"], "pipeline": output["pipeline"], "source": "ragas"},
                )
                score_count += 1
    return score_count

# Create a local review sheet that links low-score cases to their Langfuse trace IDs.
# 낮은 점수 사례를 Langfuse trace ID와 연결한 로컬 검토표를 만듭니다.
def build_case_review(outputs, score_df):
    review = pd.DataFrame([{
        "domain": output["domain"],
        "pipeline": output["pipeline"],
        "trace_id": output["trace_id"],
        "question": output["user_input"],
        "reference": output["reference"],
        "answer": output["response"],
        "retrieved_document_count": len(output["retrieved_contexts"]),
        **{metric: score_row[metric] for metric in metric_names},
    } for output, (_, score_row) in zip(outputs, score_df.iterrows())])
    return review

# Evaluate both domains, assemble the final table, and persist it for later analysis.
# 두 도메인을 평가하고 최종 표를 만든 뒤 이후 분석을 위해 저장합니다.
naive_df_korquad, advanced_df_korquad = evaluate_pair(naive_ds_korquad, advanced_ds_korquad, "korquad")
naive_df_klue, advanced_df_klue = evaluate_pair(naive_ds_klue, advanced_ds_klue, "klue")
comparison = pd.concat([
    summarize_pair(naive_df_korquad, advanced_df_korquad, "KorQuAD (wiki)"),
    summarize_pair(naive_df_klue, advanced_df_klue, "KLUE-MRC (news)"),
], ignore_index=True)
comparison[["naive", "advanced", "delta"]] = comparison[["naive", "advanced", "delta"]].round(3)
display(comparison.pivot(index="metric", columns="domain", values=["naive", "advanced", "delta"]))
comparison.to_csv(WORK_DIR / "ragas-comparison.csv", index=False, encoding="utf-8-sig")
# Persist RAGAS scores on individual traces, then export the worst cases for review.
# 개별 trace에 RAGAS 점수를 저장하고, 낮은 점수 사례를 검토할 수 있도록 내보냅니다.
score_events = sum([
    attach_ragas_scores(naive_korquad, naive_df_korquad),
    attach_ragas_scores(advanced_korquad, advanced_df_korquad),
    attach_ragas_scores(naive_klue, naive_df_klue),
    attach_ragas_scores(advanced_klue, advanced_df_klue),
])
case_review = pd.concat([
    build_case_review(naive_korquad, naive_df_korquad),
    build_case_review(advanced_korquad, advanced_df_korquad),
    build_case_review(naive_klue, naive_df_klue),
    build_case_review(advanced_klue, advanced_df_klue),
], ignore_index=True)
case_review.to_csv(WORK_DIR / "langfuse-case-review.csv", index=False, encoding="utf-8-sig")
# Show the two lowest-faithfulness cases from each domain/pipeline group.
# 도메인/파이프라인별로 Faithfulness가 가장 낮은 사례 두 개를 보여줍니다.
lowest_cases = (case_review.sort_values("faithfulness", na_position="first")
                .groupby(["domain", "pipeline"], group_keys=False).head(2))
display(lowest_cases[["domain", "pipeline", "faithfulness", "answer_relevancy", "question", "trace_id"]])
langfuse.flush()
print(f"Attached {score_events} RAGAS scores. Langfuse events flushed. Wait 15-30 seconds, then open the Traces view.")


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

naive                       advanced  \
domain            KLUE-MRC (news) KorQuAD (wiki) KLUE-MRC (news)   
metric                                                             
answer_relevancy             0.25          0.323           0.313   
context_precision            0.60          0.783           0.892   
context_recall               0.60          0.850           0.850   
faithfulness                 0.65          0.800           0.800   

                                           delta                 
domain            KorQuAD (wiki) KLUE-MRC (news) KorQuAD (wiki)  
metric                                                           
answer_relevancy           0.356           0.062          0.034  
context_precision          0.842           0.292          0.058  
context_recall             0.850           0.250          0.000  
faithfulness               0.825           0.150          0.025

,domain,pipeline,faithfulness,answer_relevancy,question,trace_id
26,korquad,advanced,0.0,0.426063,NTSC 시스템의 비월 주사 방식은 얼마의 주파수로 갱신되는가?,5477050e95913d9e2a12bc2206f9a66b
68,klue,advanced,0.0,0.044579,현대로지스틱스를 두고 현대그룹과 경쟁하는 사모펀드의 국적은?,df29625d7f75fd0d792e8a1b4b0e0fa3
25,korquad,advanced,0.0,0.000000,흥선대원군과 같이 암살을 공모한 사람은?,74dc40d0559e4d7fccfdcbf66efc191c
51,klue,naive,0.0,0.000000,입찰 내정가격이 1억 내외인 상가는 어디 블록 안에 있는가?,2bfb8f7b3ebd54dece23e358bed5c6ae
11,korquad,naive,0.0,0.020921,법원이 대형망치 쿠데타 사건의 재심을 이유로 용의자 전원을 석방한 연도는?,d20207fb7831f311b9b8e7196258f015
50,klue,naive,0.0,0.000000,쌍용자동차의 티볼리 에어가 론칭된 곳은?,1c7f34bff56f8e24c7c446f386128fc0
72,klue,advanced,0.0,0.791887,꽃발문어' 라는 브랜드명은 몇 년도에 생겼는가?,c4fbdc5d93167dc13c514fa72f341782
6,korquad,naive,0.0,0.426063,NTSC 시스템의 비월 주사 방식은 얼마의 주파수로 갱신되는가?,4e976a0376395fd33c49b92d8bc6b63b


Attached 320 RAGAS scores. Langfuse events flushed. Wait 15-30 seconds, then open the Traces view.


## 실험 흐름도

```mermaid
flowchart TD
    A[KorQuAD / KLUE-MRC 데이터 로드] --> B[문서 청크 분할 · Chroma 인덱스 구축]
    B --> C{동일한 평가 질문}

    C --> D[Naive RAG\n원 질문 검색 → Top-3 문서 → 답변]
    C --> E[Advanced RAG]
    E --> F[Multi-Query · HyDE\n질문 변형/가설 문서로 검색 확장]
    F --> G[RRF 후보 병합]
    G --> H[Cross-Encoder Reranking\nBAAI/bge-reranker-v2-m3]
    H --> I[Top-3 문서 기반 답변]
    I --> J{Self-RAG 비평\n문서 근거가 충분한가?}
    J -->|SUPPORTED| K[최종 답변]
    J -->|NOT_SUPPORTED| L[후보 수 확대 후 1회 재검색]
    L --> I

    D --> M[질문별 Langfuse trace]
    K --> M
    M --> N[질문·정답·답변·검색 문서 원문\nSelf-RAG 상태·reranker 점수 저장]
    D --> O[RAGAS 4지표 평가]
    K --> O
    O --> P[평균 비교표\nNaive vs Advanced]
    O --> Q[질문별 RAGAS 점수]
    Q --> R[동일 trace에 점수 부착\nfaithfulness / relevancy / precision / recall]
    R --> S[Langfuse에서 저점 사례 정렬]
    N --> S
    S --> T{문서에 정답 근거가 있는가?}
    T -->|없음| U[검색 문제\n검색폭·청크·임베딩·질의 확장 개선]
    T -->|있음, 답변 오류| V[생성 문제\n프롬프트·모델·거절 정책 개선]
    T -->|지표와 실제가 충돌| W[평가 점검\n짧은 정답·LLM Judge 오판 확인]
    U --> X[개선안 반영 후 재실험]
    V --> X
    W --> X
    X -. 다음 실행 .-> C
```


## 8. 결과 분석과 해석

평가 셀의 출력이 사라졌더라도 전체 파이프라인을 다시 실행할 필요는 없다. 바로 앞 평가 셀이 저장한 `ragas-comparison.csv`와 `langfuse-case-review.csv`를 읽어 비교표와 저점 사례를 복원한다. 아래 셀은 API 호출이나 RAGAS 재평가를 하지 않는다.

재실행 후에는 Langfuse의 `Tracing → Traces`에서 `faithfulness`를 오름차순으로 정렬하고, 낮은 점수 trace의 `retrieved_contexts`와 `reranker_top_scores`를 확인한다. 정답 문서가 없다면 검색 문제, 문서에 정답이 있는데 답변이 틀리면 생성 문제로 분류한다.


In [8]:
# Restore the saved summary without re-running retrieval, LLM calls, or RAGAS.
# 검색, LLM 호출, RAGAS 평가를 다시 실행하지 않고 저장된 요약을 복원합니다.
comparison = pd.read_csv(WORK_DIR / "ragas-comparison.csv")
display(comparison.pivot(index="metric", columns="domain", values=["naive", "advanced", "delta"]))
# Show saved low-score cases only after the updated evaluation cell has created the review file.
# 갱신된 평가 셀이 검토 파일을 만든 경우에만 저장된 저점 사례를 표시합니다.
case_review_path = WORK_DIR / "langfuse-case-review.csv"
if case_review_path.exists():
    case_review = pd.read_csv(case_review_path)
    display(case_review.sort_values("faithfulness")[["domain", "pipeline", "faithfulness", "answer_relevancy", "question", "trace_id"]].head(8))
else:
    print("Run sections 6 and 7 once to create langfuse-case-review.csv.")


naive                       advanced  \
domain            KLUE-MRC (news) KorQuAD (wiki) KLUE-MRC (news)   
metric                                                             
answer_relevancy             0.25          0.323           0.313   
context_precision            0.60          0.783           0.892   
context_recall               0.60          0.850           0.850   
faithfulness                 0.65          0.800           0.800   

                                           delta                 
domain            KorQuAD (wiki) KLUE-MRC (news) KorQuAD (wiki)  
metric                                                           
answer_relevancy           0.356           0.062          0.034  
context_precision          0.842           0.292          0.058  
context_recall             0.850           0.250          0.000  
faithfulness               0.825           0.150          0.025

,domain,pipeline,faithfulness,answer_relevancy,question,trace_id
26,korquad,advanced,0.0,0.426063,NTSC 시스템의 비월 주사 방식은 얼마의 주파수로 갱신되는가?,5477050e95913d9e2a12bc2206f9a66b
68,klue,advanced,0.0,0.044579,현대로지스틱스를 두고 현대그룹과 경쟁하는 사모펀드의 국적은?,df29625d7f75fd0d792e8a1b4b0e0fa3
25,korquad,advanced,0.0,0.000000,흥선대원군과 같이 암살을 공모한 사람은?,74dc40d0559e4d7fccfdcbf66efc191c
51,klue,naive,0.0,0.000000,입찰 내정가격이 1억 내외인 상가는 어디 블록 안에 있는가?,2bfb8f7b3ebd54dece23e358bed5c6ae
11,korquad,naive,0.0,0.020921,법원이 대형망치 쿠데타 사건의 재심을 이유로 용의자 전원을 석방한 연도는?,d20207fb7831f311b9b8e7196258f015
50,klue,naive,0.0,0.000000,쌍용자동차의 티볼리 에어가 론칭된 곳은?,1c7f34bff56f8e24c7c446f386128fc0
72,klue,advanced,0.0,0.791887,꽃발문어' 라는 브랜드명은 몇 년도에 생겼는가?,c4fbdc5d93167dc13c514fa72f341782
32,korquad,advanced,0.0,0.000000,유아인의 고향은?,441d572b46a00ba0ccf795124fc99e67


### 이번 재실행 결과 (도메인별 20문항)

| 지표 | KorQuAD: Naive → Advanced | KLUE-MRC: Naive → Advanced |
| --- | ---: | ---: |
| Faithfulness | 0.800 → 0.825 (+0.025) | 0.650 → 0.800 (+0.150) |
| Answer Relevancy | 0.323 → 0.356 (+0.034) | 0.250 → 0.313 (+0.062) |
| Context Precision | 0.783 → 0.842 (+0.058) | 0.600 → 0.892 (+0.292) |
| Context Recall | 0.850 → 0.850 (+0.000) | 0.600 → 0.850 (+0.250) |

### 평균 점수 해석

**KorQuAD(위키):** Advanced RAG는 Faithfulness(+0.025), Answer Relevancy(+0.034), Context Precision(+0.058)을 모두 소폭 개선했다. 반면 Context Recall은 0.850으로 동일하다. 즉 Naive RAG도 위키 문서에서 정답 근거를 찾는 능력은 이미 높았고, Advanced RAG의 주요 효과는 새 근거를 더 많이 찾기보다 여러 후보 중 질문과 가까운 문서를 위로 올려 답변 초점을 보완한 데 있다.

**KLUE-MRC(뉴스):** 개선 폭은 뉴스 도메인에서 훨씬 크다. Context Precision은 +0.292, Context Recall은 +0.250, Faithfulness는 +0.150 상승했다. Naive RAG는 뉴스 문서의 인물·기관·날짜·수치처럼 표현이 다양한 단서를 충분히 회수하지 못했지만, Multi-Query와 HyDE가 검색 표현을 넓히고 RRF와 Cross-Encoder가 후보를 다시 정렬하면서 필요한 문서를 더 많이, 더 정확하게 제공한 것으로 해석할 수 있다. 검색 문서 품질의 상승이 답변 관련성(+0.062)과 근거성으로 이어졌다.

### Langfuse 질문별 관찰

이번 실행에서는 `rag-question` 80개와 `ragas-evaluation` 2개가 기록됐고, 각 질문 trace에 검색 문서 원문, 정답, 답변, Self-RAG 상태, reranker 상위 점수, RAGAS 4개 점수가 함께 연결됐다. 따라서 평균 점수만 보는 것이 아니라 낮은 점수의 질문을 열어 검색·생성·평가 중 어디에서 문제가 생겼는지 확인할 수 있다.

**검색 실패 사례 — 유아인의 고향:** Advanced RAG는 답을 `모르는 정보입니다`로 반환했고 네 지표가 모두 0이었다. trace의 reranker 상위 점수는 약 0.068, 0.008, 0.007로 모두 매우 낮았고, 검색 문서도 배우 활동 이력만 포함했다. 즉 reranker가 정답 문서를 잘못 밀어낸 경우라기보다 후보 집합 자체에 출생지 근거가 없었던 검색 실패다. 이 유형은 검색 후보 수 확대, 청크 크기 조정, 임베딩 모델 변경 또는 고유명사 중심 질의 확장으로 개선한다.

**두 파이프라인 공통 실패 — 흥선대원군 암살 공모자:** Naive와 Advanced 모두 정답 `미우라 고로` 대신 모른다고 답했고 네 지표가 0이었다. Advanced 기법을 추가해도 코퍼스에 정답 문단이 회수되지 않으면 해결되지 않는다는 사례다. 따라서 이 문제는 생성 모델을 바꾸기보다 검색 범위를 늘리거나 문서 청크·색인 구성을 먼저 점검해야 한다.

**평가 지표 점검 사례 — 대형망치 쿠데타 재심 연도 / 현대로지스틱스 국적:** 답변과 정답은 각각 `2014년`, `일본`으로 일치했고 Context Recall도 1.0이지만, Faithfulness가 0으로 기록된 사례가 있었다. 짧은 단답형 정답에서는 LLM Judge 기반 지표가 실제 정답성과 다르게 평가될 수 있으므로, Faithfulness 0을 곧바로 모델 실패로 단정하면 안 된다. 낮은 점수 trace는 반드시 답변·정답·검색 문서를 함께 확인해야 한다는 근거다.

**Self-RAG와 HyDE:** 일부 trace에서 HyDE가 사실처럼 보이는 가설을 만들었지만 `self-rag-critique=NOT_SUPPORTED` 뒤 `retries=1`로 재검색이 수행됐다. HyDE는 답을 생성하는 장치가 아니라 검색 후보를 넓히기 위한 가설 생성 장치로 쓰여야 하며, Self-RAG 비평은 그 가설이 최종 답변의 근거로 과도하게 사용되는 것을 막는 안전장치 역할을 했다. 다만 재시도 뒤에도 비평이 실패하면 현재 구현은 마지막 답변을 반환할 수 있으므로, 최종 실패 시 `문서에서 확인할 수 없습니다`로 제한하는 정책을 다음 개선안으로 제안할 수 있다.

> 도메인별 20문항의 소규모 평가이므로 점수 차이를 확정적 성능 우위로 단정하지 않는다. 이번 실험의 결론은 Advanced RAG가 특히 KLUE-MRC 뉴스 도메인에서 검색 품질을 크게 개선했고, Langfuse 질문별 trace가 평균 점수 뒤에 숨은 검색 실패와 평가 불일치를 구분하는 데 유용했다는 것이다.
